# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GazalaNK/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm using Random Forest regression to predict the CTR residual directly, rather than converting this into classification. The underlying question is "how much is this page under-capturing clicks," which is a continuous quantity, not a binary label — keeping it continuous preserves more information for ranking. Random Forest specifically (over plain linear regression) because CTR's relationship with position, impressions, and content features likely involves non-linear interactions a linear model would miss, and Random Forest handles that without heavy tuning.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.ensemble import RandomForestRegressor

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

df_fact = con.sql("""
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

page = df_fact.groupby(["content_hash_id","client_hash_id"]).agg(
    impressions=("gsc_impressions","sum"),
    clicks=("gsc_clicks","sum"),
    avg_position=("gsc_avg_position","mean"),
).reset_index()
page["ctr"] = page["clicks"] / page["impressions"]
page["position_tier"] = pd.cut(page["avg_position"], bins=[0,3,10,20,1000], labels=["1-3","4-10","11-20","20+"])
page["tier_median_ctr"] = page.groupby("position_tier")["ctr"].transform("median")
page["ctr_residual"] = page["ctr"] - page["tier_median_ctr"]
print(page.shape)
page.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 9)


/tmp/ipykernel_3188/1997634586.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  page["tier_median_ctr"] = page.groupby("position_tier")["ctr"].transform("median")


,content_hash_id,client_hash_id,impressions,clicks,avg_position,ctr,position_tier,tier_median_ctr,ctr_residual
0,content_000005d4ced12088,client_9958f0a7ae1df715,86,0,72.854861,0.000000,20+,0.0,0.000000
1,content_00007bd2985b77c3,client_73cda7b4e4f265ea,47,0,5.269565,0.000000,4-10,0.0,0.000000
2,content_0000cd28fbda69f3,client_3ffa76342f366962,29,0,4.251282,0.000000,4-10,0.0,0.000000
3,content_0000d495bfbfb4a8,client_2094c6eb080311d5,15,0,3.333333,0.000000,4-10,0.0,0.000000
4,content_00014efc121d911d,client_08a6a72ff48e62c0,116,1,4.964683,0.008621,4-10,0.0,0.008621


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a client-grouped split, not a random or time-aware split. Client-grouped, because pages from the same client can share site-wide patterns (template, domain authority, niche) that a model could memorize rather than genuinely learn from — a random split risks leaking client identity into the "unseen" test set. Not time-aware, because this is a same-month scoring task, not predicting a future outcome, so there's no forward-looking window to protect here.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(page, groups=page["client_hash_id"]))
train, test = page.iloc[train_idx].copy(), page.iloc[test_idx].copy()
print(train.shape, test.shape)


(138310, 9) (38428, 9)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Comparing my Random Forest's ranking against my Week-4 baseline rule, on the same test split, using Precision@20 — the same metric justified in ML-03/ML-04.

In [12]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

features = ["impressions", "clicks", "avg_position"]
target_col = "ctr_residual"

# --- Clean training data ---
initial_train_rows = train.shape[0]
train = train.replace([np.inf, -np.inf], np.nan)
train = train.dropna(subset=features + [target_col]).reset_index(drop=True)
print(f"Train: removed {initial_train_rows - train.shape[0]} rows with NaN/Inf. Remaining: {train.shape[0]}")

# --- Clean testing data ---
initial_test_rows = test.shape[0]
test = test.replace([np.inf, -np.inf], np.nan)
test = test.dropna(subset=features + [target_col]).reset_index(drop=True)
print(f"Test: removed {initial_test_rows - test.shape[0]} rows with NaN/Inf. Remaining: {test.shape[0]}")

X_train, y_train = train[features], train[target_col]
X_test, y_test = test[features], test[target_col]

# --- Train ---
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# --- Predict (reset index to avoid alignment issues) ---
test = test.reset_index(drop=True)
test["model_pred"] = model.predict(X_test)

# --- Diagnostic before computing MAE ---
print("NaNs in y_test:", y_test.isna().sum())
print("NaNs in model_pred:", test["model_pred"].isna().sum())

# --- Safe MAE ---
mae = mean_absolute_error(y_test, test["model_pred"])
print("MAE:", mae)

# --- Scores ---
test["baseline_score"] = -1 * test["ctr_residual"]
test["model_score"] = -1 * test["model_pred"]

def precision_at_k(df, score_col, k=20, threshold=-0.02):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return (top_k["ctr_residual"] < threshold).mean()

comparison = pd.DataFrame({
    "method": ["baseline_rule", "random_forest"],
    "precision_at_20": [precision_at_k(test, "baseline_score"), precision_at_k(test, "model_score")]
})

Train: removed 1215 rows with NaN/Inf. Remaining: 137095
Test: removed 219 rows with NaN/Inf. Remaining: 38209
NaNs in y_test: 0
NaNs in model_pred: 0
MAE: 5.0254747722588256e-05


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.inspection import permutation_importance

result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
for i in result.importances_mean.argsort()[::-1]:
    print(f"{features[i]}: {result.importances_mean[i]:.4f}")

test["error"] = test["model_pred"] - test["ctr_residual"]
print(test.groupby("position_tier")["error"].mean())


impressions: 22.7960
clicks: 22.5142
avg_position: -0.0001
position_tier
1-3     -0.000205
4-10    -0.000021
11-20   -0.000015
20+     -0.000010
Name: error, dtype: float64


/tmp/ipykernel_3188/2179459863.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(test.groupby("position_tier")["error"].mean())


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.